# TRIBE v2 — Pipeline Benchmark

This notebook measures the wall-clock time of every major phase in the TRIBE v2 pipeline to establish baselines for optimization.

**Requirements:**
- GPU runtime (Menu → Runtime → Change runtime type → T4 GPU)
- HuggingFace account with access to [LLaMA 3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B)

**Output:** A `benchmark_results.json` file with per-phase timings, GPU info, and memory snapshots.

**Setup:** Run cell 1 (install), then restart the runtime, then run all remaining cells.

## 1. Install dependencies (run this cell, then restart runtime)

In [ ]:
!pip install "tribev2[plotting] @ git+https://github.com/kingjulio8238/tribev2.git" -q
!pip install "numpy==2.1.0" -q
!pip install "faster-whisper" -q

print("\n⚠️  Now restart the runtime: Runtime → Restart session, then run all cells BELOW this one.")

## 2. Benchmark utilities (start here after restart)

In [ ]:
import time
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass, field, asdict
from contextlib import contextmanager
import numpy as np


@dataclass
class PhaseResult:
    """Timing + memory snapshot for a single pipeline phase."""
    name: str
    elapsed_s: float = 0.0
    gpu_mem_allocated_mb: float = 0.0
    gpu_mem_reserved_mb: float = 0.0
    notes: str = ""


@dataclass
class BenchmarkResults:
    """Collects all phase results + system info."""
    gpu_name: str = ""
    gpu_vram_mb: float = 0.0
    cuda_version: str = ""
    torch_version: str = ""
    video_duration_s: float = 0.0
    video_file: str = ""
    phases: list = field(default_factory=list)

    def add(self, result: PhaseResult):
        self.phases.append(asdict(result))
        print(f"  \u2713 {result.name}: {result.elapsed_s:.2f}s"
              f"  (GPU: {result.gpu_mem_allocated_mb:.0f} MB alloc, "
              f"{result.gpu_mem_reserved_mb:.0f} MB reserved)")

    def summary(self):
        total = sum(p["elapsed_s"] for p in self.phases)
        print("\n" + "="*60)
        print(f"BENCHMARK SUMMARY  ({self.gpu_name}, {self.gpu_vram_mb:.0f} MB VRAM)")
        print("="*60)
        for p in self.phases:
            pct = (p["elapsed_s"] / total * 100) if total > 0 else 0
            bar = "\u2588" * int(pct / 2)
            print(f"  {p['name']:<35s} {p['elapsed_s']:>8.2f}s  ({pct:5.1f}%)  {bar}")
        print(f"  {'TOTAL':<35s} {total:>8.2f}s")
        print("="*60)

    def to_dict(self):
        return asdict(self)


def gpu_mem_snapshot():
    """Return (allocated_mb, reserved_mb) on the current CUDA device."""
    try:
        import torch
        if torch.cuda.is_available():
            return (
                torch.cuda.memory_allocated() / 1e6,
                torch.cuda.memory_reserved() / 1e6,
            )
    except Exception:
        pass
    return (0.0, 0.0)


@contextmanager
def timed_phase(bench: BenchmarkResults, name: str, notes: str = ""):
    """Context manager that times a phase and records GPU memory."""
    print(f"\n\u25b6 {name} ...")
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    alloc, reserved = gpu_mem_snapshot()
    bench.add(PhaseResult(
        name=name,
        elapsed_s=round(elapsed, 3),
        gpu_mem_allocated_mb=round(alloc, 1),
        gpu_mem_reserved_mb=round(reserved, 1),
        notes=notes,
    ))


bench = BenchmarkResults()

# ── Pre-generate fsaverage5 mesh (one-time, cached for export) ──
import nibabel as nib
from nilearn.datasets import fetch_surf_fsaverage

MESH_CACHE = Path("./mesh_cache")
if not (MESH_CACHE / "vertices.bin").exists():
    MESH_CACHE.mkdir(exist_ok=True)
    fsaverage = fetch_surf_fsaverage("fsaverage5")
    all_coords, all_faces, all_sulc = [], [], []
    vertex_offset = 0
    GAP = 3.0
    for hemi in ("left", "right"):
        pial = nib.load(getattr(fsaverage, f"pial_{hemi}")).darrays
        pial_coords, faces = pial[0].data, pial[1].data
        infl_coords = nib.load(getattr(fsaverage, f"infl_{hemi}")).darrays[0].data
        coords = 0.5 * pial_coords + 0.5 * infl_coords
        if hemi == "left":
            coords[:, 0] -= coords[:, 0].max() + GAP
        else:
            coords[:, 0] -= coords[:, 0].min() - GAP
        sulc = nib.load(getattr(fsaverage, f"sulc_{hemi}")).darrays[0].data
        all_coords.append(coords)
        all_faces.append(faces + vertex_offset)
        all_sulc.append(sulc)
        vertex_offset += coords.shape[0]
    vertices = np.concatenate(all_coords).astype(np.float32)
    faces_arr = np.concatenate(all_faces).astype(np.uint32)
    sulcal = np.concatenate(all_sulc).astype(np.float32)
    vertices.tofile(str(MESH_CACHE / "vertices.bin"))
    faces_arr.tofile(str(MESH_CACHE / "faces.bin"))
    sulcal.tofile(str(MESH_CACHE / "sulcal_depth.bin"))
    # HCP parcellation
    try:
        from tribev2.utils import get_hcp_labels
        labels_dict = get_hcp_labels(mesh="fsaverage5", combine=False, hemi="both")
        roi_names = ["unknown"] + sorted(labels_dict.keys())
        roi_map = {n: i for i, n in enumerate(roi_names)}
        parc = np.zeros(vertices.shape[0], dtype=np.uint16)
        for name, verts in labels_dict.items():
            for v in verts:
                if v < parc.shape[0]:
                    parc[int(v)] = roi_map[name]
        parc.tofile(str(MESH_CACHE / "parcellation.bin"))
        with open(MESH_CACHE / "roi_names.json", "w") as f:
            json.dump(roi_names, f)
    except Exception as e:
        print(f"  Parcellation skipped: {e}")
    print(f"  Mesh pre-generated: {vertices.shape[0]} vertices, {faces_arr.shape[0]} faces")
else:
    print("  Mesh cache exists, skipping generation")

print("Benchmark utilities loaded.")

## 3. Collect system info

In [ ]:
import torch

bench.torch_version = torch.__version__
bench.cuda_version = torch.version.cuda or "N/A"

if torch.cuda.is_available():
    bench.gpu_name = torch.cuda.get_device_name(0)
    bench.gpu_vram_mb = torch.cuda.get_device_properties(0).total_memory / 1e6
    print(f"GPU: {bench.gpu_name}")
    print(f"VRAM: {bench.gpu_vram_mb:.0f} MB")
    print(f"CUDA: {bench.cuda_version}")
else:
    print("WARNING: No GPU detected! Benchmarks will be very slow.")

print(f"PyTorch: {bench.torch_version}")

# Also capture nvidia-smi for the record
!nvidia-smi

## 3. Authenticate with HuggingFace

In [ ]:
from huggingface_hub import login
login()

## 4. Choose your video

**Option A:** Upload to Google Drive, then set the path below (fastest)

**Option B:** Download from a URL (set `VIDEO_MODE = "url"`)

Supported formats: `.mp4`, `.avi`, `.mkv`, `.mov`, `.webm`. Audio required for full pipeline.

In [ ]:
from tribev2.demo_utils import download_file
from pathlib import Path

# ──────────────────────────────────────────────
# Choose one: "drive" or "url"
VIDEO_MODE = "drive"
# For drive mode: upload your video to Google Drive, then set the path
DRIVE_VIDEO_PATH = "/content/drive/MyDrive/your_video.mp4"
# For url mode: set the download URL
VIDEO_URL = "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"
# ──────────────────────────────────────────────

CACHE_FOLDER = Path("./cache")
CACHE_FOLDER.mkdir(exist_ok=True)

if VIDEO_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    src = Path(DRIVE_VIDEO_PATH)
    if not src.exists():
        raise FileNotFoundError(f"Video not found at {src}. Upload it to Google Drive and update DRIVE_VIDEO_PATH.")
    video_path = CACHE_FOLDER / src.name
    import shutil
    shutil.copy2(str(src), str(video_path))
    print(f"Copied from Drive: {src.name}")
else:
    video_path = CACHE_FOLDER / VIDEO_URL.split("/")[-1]
    with timed_phase(bench, "2. Download video"):
        download_file(VIDEO_URL, video_path)

# Pre-convert to compressed H.264 mp4 (speeds up frame extraction)
import subprocess
_src_ext = video_path.suffix.lower()
_mp4_path = video_path.with_suffix(".mp4")
if _src_ext != ".mp4" or video_path.stat().st_size > 100 * 1e6:
    print(f"Pre-converting {video_path.name} ({video_path.stat().st_size / 1e6:.0f} MB) to compressed mp4...")
    _r = subprocess.run(
        ["ffmpeg", "-y", "-i", str(video_path), "-c:v", "libx264", "-crf", "23",
         "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart",
         str(_mp4_path)],
        capture_output=True, text=True
    )
    if _r.returncode == 0:
        _old_mb = video_path.stat().st_size / 1e6
        _new_mb = _mp4_path.stat().st_size / 1e6
        video_path = _mp4_path
        print(f"  Converted: {_old_mb:.0f} MB → {_new_mb:.0f} MB ({_old_mb/_new_mb:.1f}x smaller)")
    else:
        print(f"  ffmpeg failed, using original: {_r.stderr[:200]}")

# Record video duration
from moviepy import VideoFileClip
clip = VideoFileClip(str(video_path))
bench.video_duration_s = clip.duration
bench.video_file = video_path.name
print(f"Video: {video_path.name}, {clip.duration:.1f}s, {clip.size}")
clip.close()


## 4b. Video brief (for effectiveness report)

Describe the video's objective so the AI can evaluate whether brain/emotional responses align with intent.
Leave empty to skip the effectiveness report.

In [ ]:
VIDEO_BRIEF = {
    "title": "Bud Light Super Bowl Ad",
    "objective": "Drive brand awareness and positive sentiment for Bud Light among young adult males",
    "target_audience": "21-35 year old males, sports fans, social drinkers",
    "intended_emotions": ["Excitement", "Humor/Joy", "Social connection"],
    "key_moments": "Wedding scene, unexpected twist, product reveal at end",
    "success_criteria": "High engagement during product reveal, positive emotional arc, strong memorability",
}

if VIDEO_BRIEF.get("title"):
    print(f"Brief loaded: {VIDEO_BRIEF['title']}")
    print(f"  Objective: {VIDEO_BRIEF['objective']}")
    print(f"  Audience: {VIDEO_BRIEF['target_audience']}")
else:
    print("No brief provided — effectiveness report will be skipped")


## 5. Load model (checkpoint download + init)

In [ ]:
from tribev2.demo_utils import TribeModel
import uuid

# Use a unique cache folder per run to prevent feature cache hits
RUN_ID = uuid.uuid4().hex[:8]
RUN_CACHE = Path(f"./cache_{RUN_ID}")
RUN_CACHE.mkdir(exist_ok=True)

# Copy video to the new cache folder
import shutil
shutil.copy2(str(video_path), str(RUN_CACHE / "sintel_trailer.mp4"))

with timed_phase(bench, "3. Model load (checkpoint + init)",
                 notes="Downloads ~709MB checkpoint on first run, inits config"):
    model = TribeModel.from_pretrained(
        "facebook/tribev2",
        cache_folder=RUN_CACHE,
        device="auto",
    )
print(f"Model loaded successfully (cache: {RUN_CACHE})")

## 5b. Apply optimizations (toggle ON/OFF)

Optimizations auto-detect GPU capabilities (BF16 on A100/H100, FP16 on T4).
`reduce-overhead` compile mode used on high-VRAM GPUs, `default` on T4.

In [ ]:
# ──────────────────────────────────────────────
# Toggle this flag to compare optimized vs baseline
ENABLE_OPTIMIZATIONS = True
# V-JEPA2 frame count: 64 (default) or 32 (subsampled)
VJEPA2_NUM_FRAMES = 32
# ──────────────────────────────────────────────

if ENABLE_OPTIMIZATIONS:
    import torch
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True

    # Auto-detect GPU capabilities
    _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    _is_ampere_plus = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    _high_vram = _vram_gb > 20

    _dtype = torch.bfloat16 if _is_ampere_plus else torch.float16
    _dtype_name = "BF16" if _is_ampere_plus else "FP16"
    _compile_mode = "reduce-overhead" if _high_vram else "default"

    print(f"  GPU: {_gpu_name} ({_vram_gb:.0f} GB)")
    print(f"  Precision: {_dtype_name} | Compile: {_compile_mode} | cuDNN benchmark: ON")

    # ================================================================
    # 1. V-JEPA2: half-precision + torch.compile + no_grad + subsample
    # ================================================================
    from neuralset.extractors.video import _HFVideoModel, _fix_pixel_values

    if not getattr(_HFVideoModel, '_optimized', False):
        _HFVideoModel._original_init = _HFVideoModel.__init__
        _HFVideoModel._original_predict = _HFVideoModel.predict
        _in_init = False

        def _patched_hfv_init(self, model_name, pretrained=True, layer_type="", num_frames=None):
            global _in_init
            if _in_init:
                return _HFVideoModel._original_init(self, model_name, pretrained, layer_type, num_frames)
            _in_init = True
            try:
                if "vjepa2" in model_name.lower() and num_frames is None:
                    num_frames = VJEPA2_NUM_FRAMES
                    print(f"  V-JEPA2: subsampling to {num_frames} frames (from 64)")
                _HFVideoModel._original_init(self, model_name, pretrained, layer_type, num_frames)
                if "vjepa2" in model_name.lower() and torch.cuda.is_available():
                    self.model = self.model.to(_dtype)
                    self.model.requires_grad_(False)
                    print(f"  V-JEPA2: {_dtype_name} + no_grad applied")
                    try:
                        self.model = torch.compile(self.model, backend="inductor", mode=_compile_mode)
                        print(f"  V-JEPA2: torch.compile ({_compile_mode}) applied")
                    except Exception as e:
                        print(f"  V-JEPA2: torch.compile skipped ({e})")
            finally:
                _in_init = False

        _HFVideoModel.__init__ = _patched_hfv_init

        def _patched_predict(self, images, audio=None):
            kwargs = {"text": "", "return_tensors": "pt"}
            field = "images"
            if "xclip" in self.model_name:
                field = "videos"
            elif "llava" in self.model_name.lower():
                field = "videos"
                kwargs["text"] = self.layer_type
            elif "vjepa2" in self.model_name:
                field = "videos"
                del kwargs["text"]
            elif "Phi-4" in self.model_name:
                import PIL
                images = [PIL.Image.fromarray(img) for img in images]
                field = "images"
                prompt = "<|user|>"
                for i in range(1, len(images) + 1):
                    prompt += f"<|image_{i}|>"
                if audio is not None:
                    kwargs["audios"] = [(audio.to_soundarray(), audio.fps)]
                    prompt += "<|audio_1|>"
                prompt += "<|end|><|assistant|>"
                kwargs["text"] = prompt
            kwargs[field] = list(images)
            inputs = self.processor(**kwargs)
            _fix_pixel_values(inputs)
            inputs = inputs.to(self.model.device)
            model_dtype = next(self.model.parameters()).dtype
            if model_dtype in (torch.float16, torch.bfloat16):
                for key in inputs:
                    if isinstance(inputs[key], torch.Tensor) and inputs[key].is_floating_point():
                        inputs[key] = inputs[key].to(model_dtype)
            with torch.inference_mode():
                pred = self.model(**inputs)
            return pred

        _HFVideoModel.predict = _patched_predict
        _HFVideoModel._optimized = True

    # ================================================================
    # 2. FASTER-WHISPER: Replace WhisperX subprocess with faster-whisper
    # ================================================================
    from tribev2.eventstransforms import ExtractWordsFromAudio
    import pandas as pd

    if not getattr(ExtractWordsFromAudio, '_faster_whisper_patched', False):
        @staticmethod
        def _fast_transcript(wav_filename, language):
            from faster_whisper import WhisperModel
            language_codes = dict(english="en", french="fr", spanish="es", dutch="nl", chinese="zh")
            lang = language_codes.get(language, "en")

            device = "cuda" if torch.cuda.is_available() else "cpu"
            compute_type = "float16" if device == "cuda" else "float32"

            model = WhisperModel("large-v3", device=device, compute_type=compute_type)
            segments, info = model.transcribe(
                str(wav_filename), language=lang, beam_size=5,
                word_timestamps=True, vad_filter=True,
            )
            words = []
            for i, segment in enumerate(segments):
                sentence = segment.text.replace('"', "").strip()
                if segment.words:
                    for word in segment.words:
                        words.append({
                            "text": word.word.replace('"', "").strip(),
                            "start": word.start,
                            "duration": word.end - word.start,
                            "sequence_id": i,
                            "sentence": sentence,
                        })
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return pd.DataFrame(words)

        ExtractWordsFromAudio._get_transcript_from_audio = _fast_transcript
        ExtractWordsFromAudio._faster_whisper_patched = True
        print("  WhisperX replaced with faster-whisper (large-v3)")

    print(f"\n\u2705 Optimizations ENABLED ({_dtype_name} + compile-{_compile_mode} + {VJEPA2_NUM_FRAMES}-frame + faster-whisper)")
else:
    print("\u26a0\ufe0f  Optimizations DISABLED (baseline mode)")

## 6. Build events (audio extraction + WhisperX transcription)

In [ ]:
with timed_phase(bench, "4. Build events (audio + WhisperX)",
                 notes="ExtractAudioFromVideo, ChunkEvents, WhisperX, AddText/Context"):
    df = model.get_events_dataframe(video_path=video_path)

print(f"Events: {len(df)} rows")
print(f"Types: {df['type'].value_counts().to_dict()}")
display(df.head(8)[["type", "start", "duration", "text", "context"]])

## 7. Feature extraction (per-modality, instrumented)

This is where the bulk of compute happens. We call `get_loaders` which internally runs `extractor.prepare(events)` for each modality. We monkey-patch to time each extractor individually.

In [ ]:
import gc
from neuralset.events.utils import standardize_events
from neuralset.events.etypes import EventTypesHelper
import neuralset as ns
import numpy as np

# We replicate the get_loaders logic but instrument each extractor.prepare() call
events = standardize_events(df)

# Build extractors dict (same logic as Data.get_loaders)
extractors = {}
for modality in model.data.features_to_use:
    extractors[modality] = getattr(model.data, f"{modality}_feature")
extractors["subject_id"] = model.data.subject_id

# Add dummy trigger events (same as get_loaders)
dummy_events = []
for timeline_name, timeline in events.groupby("timeline"):
    split = "all"
    dummy_event = {
        "type": "CategoricalEvent",
        "timeline": timeline_name,
        "start": timeline.start.min(),
        "duration": timeline.stop.max() - timeline.start.min(),
        "split": split,
        "subject": timeline.subject.unique()[0],
    }
    dummy_events.append(dummy_event)
import pandas as pd
events_with_triggers = pd.concat([events, pd.DataFrame(dummy_events)])
events_with_triggers = standardize_events(events_with_triggers)

# Remove extractors with no matching events
to_remove = set()
for name, ext in extractors.items():
    event_types = EventTypesHelper(ext.event_types).names
    if not any(et in events_with_triggers.type.unique() for et in event_types):
        to_remove.add(name)
for name in to_remove:
    del extractors[name]
    print(f"  Skipping extractor '{name}' (no matching events)")

print(f"\nExtractors to benchmark: {list(extractors.keys())}")

In [ ]:
from tribev2.main import _free_extractor_model
import gc
from concurrent.futures import ThreadPoolExecutor

# Move the tribev2 brain model to CPU during feature extraction
if model._model is not None:
    model._model.cpu()
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Moved brain model to CPU (freed ~710 MB GPU)")

def run_extractor(name, extractor, events):
    """Run a single extractor and return timing info."""
    start = time.perf_counter()
    extractor.prepare(events)
    elapsed = time.perf_counter() - start
    alloc, reserved = gpu_mem_snapshot()
    return name, elapsed, alloc, reserved

# Separate extractors into groups: parallel (text+audio) and sequential (video, subject_id)
parallel_names = {"text", "audio"}
parallel_extractors = {n: e for n, e in extractors.items() if n in parallel_names}
sequential_extractors = {n: e for n, e in extractors.items() if n not in parallel_names}

# Run text + audio in parallel (they're independent)
if len(parallel_extractors) >= 2:
    print(f"\n▶ 5. Feature extraction: text + audio (parallel) ...")
    start_parallel = time.perf_counter()
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {
            executor.submit(run_extractor, n, e, events_with_triggers): n
            for n, e in parallel_extractors.items()
        }
        for future in futures:
            name, elapsed, alloc, reserved = future.result()
            print(f"    {name}: {elapsed:.2f}s (GPU: {alloc:.0f} MB alloc)")
    elapsed_parallel = time.perf_counter() - start_parallel
    alloc, reserved = gpu_mem_snapshot()
    bench.add(PhaseResult(
        name="5. Feature extraction: text+audio (parallel)",
        elapsed_s=round(elapsed_parallel, 3),
        gpu_mem_allocated_mb=round(alloc, 1),
        gpu_mem_reserved_mb=round(reserved, 1),
        notes="text and audio run concurrently",
    ))
    # Free both models
    for n, e in parallel_extractors.items():
        _free_extractor_model(e)
    gc.collect()
    torch.cuda.empty_cache()
else:
    # Fallback: run individually
    for name, extractor in parallel_extractors.items():
        with timed_phase(bench, f"5. Feature extraction: {name}",
                         notes=f"extractor type: {type(extractor).__name__}"):
            extractor.prepare(events_with_triggers)
        _free_extractor_model(extractor)
        gc.collect()
        torch.cuda.empty_cache()

# Run remaining extractors sequentially (video needs full GPU)
for name, extractor in sequential_extractors.items():
    phase_name = f"5. Feature extraction: {name}"
    notes = f"extractor type: {type(extractor).__name__}"
    with timed_phase(bench, phase_name, notes=notes):
        extractor.prepare(events_with_triggers)
    if name != "subject_id":
        _free_extractor_model(extractor)
        gc.collect()
        torch.cuda.empty_cache()

print("\nAll feature extraction complete.")

## 8. Build dataloader

In [ ]:
with timed_phase(bench, "6. Build dataloader",
                 notes="Segment creation + SegmentDataset init"):
    TR = model.data.TR
    segments = ns.segments.list_segments(
        events_with_triggers,
        triggers=events_with_triggers.type == "CategoricalEvent",
        stride=(model.data.duration_trs - model.data.overlap_trs_train) * TR,
        duration=model.data.duration_trs * TR,
        stride_drop_incomplete=model.data.stride_drop_incomplete,
    )
    dataset = ns.dataloader.SegmentDataset(
        extractors=extractors,
        segments=segments,
        remove_incomplete_segments=False,
    )
    loader = dataset.build_dataloader(
        shuffle=False,
        num_workers=model.data.num_workers,
        batch_size=model.data.batch_size,
    )

print(f"Dataloader: {len(loader)} batches, batch_size={model.data.batch_size}")

from einops import rearrange
from tqdm import tqdm

# Move brain model back to GPU for inference
brain_model = model._model
if torch.cuda.is_available() and next(brain_model.parameters()).device.type == "cpu":
    brain_model.cuda()

with timed_phase(bench, "7. Model inference (forward pass)",
                 notes="Transformer forward pass over all batches"):
    preds_list, all_segments = [], []
    n_samples, n_kept = 0, 0
    with torch.inference_mode():
        for batch in tqdm(loader):
            batch = batch.to(brain_model.device)
            batch_segments = []
            for segment in batch.segments:
                for t in np.arange(0, segment.duration - 1e-2, TR):
                    batch_segments.append(
                        segment.copy(offset=t, duration=TR)
                    )
            keep = np.array([len(s.ns_events) > 0 for s in batch_segments])
            n_kept += keep.sum()
            n_samples += len(batch_segments)
            batch_segments = [s for i, s in enumerate(batch_segments) if keep[i]]
            y_pred = brain_model(batch).detach().cpu().numpy()
            y_pred = rearrange(y_pred, "b d t -> (b t) d")[keep]
            preds_list.append(y_pred)
            all_segments.extend(batch_segments)

    preds = np.concatenate(preds_list)

print(f"Predictions: {preds.shape} ({n_kept}/{n_samples} segments kept)")

In [ ]:
from einops import rearrange
from tqdm import tqdm

brain_model = model._model

with timed_phase(bench, "7. Model inference (forward pass)",
                 notes="Transformer forward pass over all batches"):
    preds_list, all_segments = [], []
    n_samples, n_kept = 0, 0
    with torch.inference_mode():
        for batch in tqdm(loader):
            batch = batch.to(brain_model.device)
            batch_segments = []
            for segment in batch.segments:
                for t in np.arange(0, segment.duration - 1e-2, TR):
                    batch_segments.append(
                        segment.copy(offset=t, duration=TR)
                    )
            keep = np.array([len(s.ns_events) > 0 for s in batch_segments])
            n_kept += keep.sum()
            n_samples += len(batch_segments)
            batch_segments = [s for i, s in enumerate(batch_segments) if keep[i]]
            y_pred = brain_model(batch).detach().cpu().numpy()
            y_pred = rearrange(y_pred, "b d t -> (b t) d")[keep]
            preds_list.append(y_pred)
            all_segments.extend(batch_segments)

    preds = np.concatenate(preds_list)

print(f"Predictions: {preds.shape} ({n_kept}/{n_samples} segments kept)")

## 10. Normalization

In [ ]:
from tribev2.plotting.utils import robust_normalize

with timed_phase(bench, "8. Normalization (robust_normalize)",
                 notes="Per-vertex 99th percentile normalization"):
    preds_norm = robust_normalize(preds, percentile=99).astype(np.float32)

print(f"Normalized predictions: {preds_norm.shape}, "
      f"range [{preds_norm.min():.3f}, {preds_norm.max():.3f}]")

## 11. Export (mesh + predictions + stimulus metadata)

In [ ]:
import shutil

OUTPUT_DIR = Path("./viewer_data")

with timed_phase(bench, "9a. Export mesh",
                 notes="Copy pre-generated mesh from cache"):
    mesh_dir = OUTPUT_DIR / "mesh"
    mesh_dir.mkdir(parents=True, exist_ok=True)
    # Copy pre-generated mesh files from cache
    for f in MESH_CACHE.glob("*"):
        shutil.copy2(str(f), str(mesh_dir / f.name))

print(f"  Mesh copied from cache ({len(list(mesh_dir.glob('*')))} files)")

In [ ]:
with timed_phase(bench, "9b. Export predictions",
                 notes="Binary predictions + metadata JSON"):
    pred_dir = OUTPUT_DIR / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    preds_norm.tofile(str(pred_dir / "predictions.bin"))

    n_timesteps = preds_norm.shape[0]
    metadata = {
        "nTimesteps": int(n_timesteps),
        "nVertices": int(preds_norm.shape[1]),
        "trSeconds": 1.0,
        "vmin": 0.5,
        "alphaScale": 0.2,
    }
    with open(pred_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

print(f"  Predictions: {preds_norm.shape}")

In [ ]:
with timed_phase(bench, "9c. Export stimulus metadata",
                 notes="segments.json + video copy + thumbnails"):
    stim_dir = OUTPUT_DIR / "stimulus"
    stim_dir.mkdir(parents=True, exist_ok=True)

    # Segments JSON
    segments_data = []
    for seg in all_segments:
        entry = {"time": float(seg.start), "hasEvents": len(seg.ns_events) > 0, "words": []}
        for ev in seg.ns_events:
            if ev.__class__.__name__ == "Word":
                entry["words"].append({
                    "text": str(ev.text),
                    "start": float(ev.start),
                    "end": float(ev.start + ev.duration),
                })
        segments_data.append(entry)
    with open(stim_dir / "segments.json", "w") as f:
        json.dump(segments_data, f, indent=2)

    # Re-encode video to compressed H.264 mp4 (handles .mov, .avi, etc.)
    import subprocess
    _ffmpeg_result = subprocess.run(
        ["ffmpeg", "-y", "-i", str(video_path), "-c:v", "libx264", "-crf", "23",
         "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart",
         str(stim_dir / "media.mp4")],
        capture_output=True, text=True
    )
    if _ffmpeg_result.returncode != 0:
        print(f"  ffmpeg failed, falling back to copy: {_ffmpeg_result.stderr[:200]}")
        shutil.copy2(str(video_path), str(stim_dir / "media.mp4"))
    else:
        _src_mb = video_path.stat().st_size / 1e6
        _dst_mb = (stim_dir / "media.mp4").stat().st_size / 1e6
        print(f"  Video re-encoded: {_src_mb:.0f} MB → {_dst_mb:.0f} MB")

    # Thumbnails
    thumb_dir = stim_dir / "thumbnails"
    thumb_dir.mkdir(exist_ok=True)
    try:
        from moviepy import VideoFileClip
        from PIL import Image
        clip = VideoFileClip(str(video_path))
        for i, seg in enumerate(all_segments):
            t = seg.start
            if 0 <= t < clip.duration:
                frame = clip.get_frame(t)
                Image.fromarray(frame).save(str(thumb_dir / f"frame_{i:05d}.jpg"), "JPEG", quality=80)
        clip.close()
    except Exception as e:
        print(f"  Thumbnails skipped: {e}")

print(f"  Segments: {len(segments_data)}, video copied, thumbnails extracted")

## 11b. Emotion analysis via Groq

Sends per-lobe activation timeseries to Groq LLM to generate per-timestep emotion scores.
Requires a Groq API key. Set `SKIP_EMOTIONS = True` to skip this step.

## 11c. Effectiveness report via Groq

Evaluates whether the video achieved its stated objectives based on brain region activity and emotional responses.
Requires both `VIDEO_BRIEF` and `emotions.json` to be present.

In [ ]:
SKIP_REPORT = False

if not SKIP_REPORT and GROQ_API_KEY and VIDEO_BRIEF.get('title') and (OUTPUT_DIR / 'emotions.json').exists():
    import json as _json
    import requests

    # Load the emotions data we just generated
    _emotions = _json.loads((OUTPUT_DIR / 'emotions.json').read_text())

    # Load segments for transcript context
    _segments = _json.loads((OUTPUT_DIR / 'stimulus' / 'segments.json').read_text())
    _transcript = []
    for seg in _segments:
        for w in seg.get('words', []):
            _transcript.append(f"{w['start']:.1f}s: {w['text']}")
    _transcript_str = ' | '.join(_transcript[:200])  # cap for token limits

    # Compute per-lobe summary stats
    _lobe_summary = {}
    for lobe in lobe_names:
        vals = lobe_timeseries[lobe]
        _lobe_summary[lobe] = {
            'mean': round(sum(vals)/len(vals), 3),
            'peak': round(max(vals), 3),
            'peak_time': vals.index(max(vals)),
        }

    # Compute emotion summary stats
    _emotion_summary = {}
    _emotion_names = list(_emotions[0]['emotions'].keys())
    for emo in _emotion_names:
        vals = [e['emotions'][emo] for e in _emotions]
        _emotion_summary[emo] = {
            'mean': round(sum(vals)/len(vals), 3),
            'peak': round(max(vals), 3),
            'peak_time': vals.index(max(vals)),
        }

    prompt = f"""You are a neuro-creative strategist. Analyze whether a video advertisement achieved its stated objectives based on brain imaging predictions and emotional response data.

## Video Brief
{_json.dumps(VIDEO_BRIEF, indent=2)}

## Brain Region Activity (per-lobe summary)
{_json.dumps(_lobe_summary, indent=2)}

## Brain Region Timeseries (0-1 scale, 1 TR/second)
{_json.dumps(lobe_timeseries)}

## Emotional Response Summary
{_json.dumps(_emotion_summary, indent=2)}

## Emotional Response Timeseries
{_json.dumps(_emotions)}

## Transcript
{_transcript_str}

Generate a structured effectiveness report. Return ONLY valid JSON with this exact schema:
{{
  "title": "string - video title",
  "overallScore": number 0-100,
  "summary": "string - 2-3 sentence executive summary of effectiveness",
  "emotionalArc": {{
    "intended": ["list of intended emotions from brief"],
    "actual": ["list of dominant actual emotions observed"],
    "alignment": number 0.0-1.0
  }},
  "keyMoments": [
    {{
      "time": number (start second),
      "endTime": number (end second),
      "label": "string - what happens",
      "engagement": number 0.0-1.0,
      "dominantEmotions": ["top 2 emotions"],
      "alignsWithObjective": boolean,
      "insight": "string - why this matters for the objective"
    }}
  ],
  "brainInsights": ["string - 3-5 key neuroscience-grounded observations"],
  "recommendations": ["string - 3-5 actionable recommendations to improve effectiveness"]
}}

Return raw JSON only, no markdown, no explanation."""

    print('Generating effectiveness report via Groq...')
    resp = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': 'llama-3.3-70b-versatile',
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0.3,
            'max_tokens': 4096,
        },
        timeout=120,
    )
    resp.raise_for_status()
    raw = resp.json()['choices'][0]['message']['content']

    try:
        report = _json.loads(raw)
    except _json.JSONDecodeError:
        # Strip markdown fences if LLM wrapped the JSON
        raw = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
        report = _json.loads(raw)
    report_path = OUTPUT_DIR / 'report.json'
    with open(report_path, 'w') as f:
        _json.dump(report, f, indent=2)

    print(f"\nEffectiveness Score: {report['overallScore']}/100")
    print(f"Summary: {report['summary']}")
    print(f"Key Moments: {len(report['keyMoments'])}")
    print(f"Recommendations: {len(report['recommendations'])}")
    print(f"Saved to {report_path}")

elif not VIDEO_BRIEF.get('title'):
    print('Skipping report — no VIDEO_BRIEF provided')
elif not GROQ_API_KEY:
    print('Skipping report — no GROQ_API_KEY set')
else:
    print('Skipping report — emotions.json not found (run emotion analysis first)')


In [ ]:
SKIP_EMOTIONS = False
GROQ_API_KEY = ""  # paste your Groq API key here

if not SKIP_EMOTIONS and GROQ_API_KEY:
    import json as _json
    from pathlib import Path as _Path

    # 1. Compute per-lobe activations for all timesteps
    from tribev2.plotting.utils import robust_normalize as _robust_normalize
    from neuralset.events.utils import standardize_events as _se

    # Re-use preds_norm and all_segments from earlier cells
    n_ts = preds_norm.shape[0]
    lobe_names = ['Frontal', 'Temporal', 'Parietal', 'Occipital', 'Insular', 'Cingulate']

    # Build lobe activations from the region activity logic
    from tribev2.utils import get_hcp_labels
    labels_dict = get_hcp_labels(mesh='fsaverage5', combine=False, hemi='both')

    # Map ROIs to lobes (simplified version of roiGroups.ts)
    LOBE_ROIS = {
        'Frontal': ['IFSp','IFSa','IFJa','IFJp','46','p47r','a47r','47s','47m','44','45',
                    '6a','6d','6ma','6mp','6r','6v','8Av','8Ad','8BL','8C','9a','9m','9p',
                    'i6-8','s6-8','SFL','SCEF','p9-46v','a9-46v','9-46d','p10p','a10p',
                    '10d','10r','10v','10pp','11l','13l','OFC','pOFC','a24','p24','s32',
                    'a32pr','p32pr','d32','p32','25'],
        'Temporal': ['A1','A4','A5','LBelt','MBelt','PBelt','RI','STSdp','STSda','STSva',
                     'STSvp','STGa','TE1a','TE1m','TE1p','TE2a','TE2p','TGd','TGv',
                     'TF','EC','PeEc','PHA1','PHA2','PHA3','H'],
        'Parietal': ['1','2','3a','3b','5L','5m','5mv','7AL','7Am','7PC','7PL','7Pm',
                     'AIP','IP0','IP1','IP2','IPS1','LIPd','LIPv','MIP','VIP',
                     'PFm','PF','PFop','PFt','PGi','PGs','PGp',
                     'DVT','ProS','POS1','POS2','PCV','7m','31a','31pd','31pv','d23ab','v23ab'],
        'Occipital': ['V1','V2','V3','V3A','V3B','V3CD','V4','V4t','V6','V6A','V7','V8',
                      'VMV1','VMV2','VMV3','VVC','FST','FFC','PIT','LO1','LO2','LO3',
                      'MST','MT','PH','PHT'],
        'Insular': ['Ig','MI','Pol1','Pol2','FOP1','FOP2','FOP3','FOP4','FOP5',
                    'AVI','AAIC','Pir','52'],
        'Cingulate': ['a24pr','p24pr','33pr','RSC','23c','23d',
                      'd23ab','v23ab','POS2','DVT','ProS'],
    }

    # Build vertex index sets per lobe
    roi_to_lobe = {}
    for lobe, rois in LOBE_ROIS.items():
        for roi in rois:
            roi_to_lobe.setdefault(roi, lobe)

    lobe_vertices = {lobe: [] for lobe in lobe_names}
    for roi_name, vertices in labels_dict.items():
        clean = roi_name.split('_')[0] if '_' in roi_name else roi_name
        lobe = roi_to_lobe.get(clean)
        if lobe:
            lobe_vertices[lobe].extend(vertices)

    # Compute per-lobe mean activation per timestep
    lobe_timeseries = {}
    for lobe in lobe_names:
        idxs = [int(v) for v in lobe_vertices[lobe] if int(v) < preds_norm.shape[1]]
        if idxs:
            lobe_timeseries[lobe] = [round(float(preds_norm[t, idxs].mean()), 4) for t in range(n_ts)]
        else:
            lobe_timeseries[lobe] = [0.0] * n_ts

    print(f'Computed lobe timeseries: {n_ts} timesteps x {len(lobe_names)} lobes')

    # 2. Send to Groq
    import requests

    prompt = f"""You are a neuroscience expert. Given brain region activation timeseries data from an fMRI prediction model watching a video, analyze the emotional response at each timestep.

Brain region activations (0-1 scale, {n_ts} timesteps at 1 TR/second):
{_json.dumps(lobe_timeseries)}

Map these brain activations to the following emotions based on neuroscience:
- Engagement: driven by Frontal + Temporal activity (attention + language processing)
- Tension: driven by Insular + Cingulate activity (emotional arousal + conflict monitoring)
- Wonder: driven by Occipital + Parietal activity (visual novelty + spatial processing)
- Empathy: driven by Temporal + Insular activity (social cognition + emotional resonance)
- Excitement: driven by Frontal + Occipital activity (reward anticipation + visual stimulation)
- Unease: driven by Cingulate + Insular activity (anxiety + interoceptive awareness)

Return ONLY a JSON array with {n_ts} objects, one per timestep. Each object must have:
- "time": the timestep index (0-based)
- "emotions": object with keys Engagement, Tension, Wonder, Empathy, Excitement, Unease — each a float 0.0 to 1.0

Return raw JSON only, no markdown, no explanation."""

    print('Sending to Groq API...')
    resp = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': 'llama-3.3-70b-versatile',
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0.3,
            'max_tokens': 8192,
        },
        timeout=120,
    )
    resp.raise_for_status()
    raw = resp.json()['choices'][0]['message']['content']

    # 3. Parse and save
    emotions = _json.loads(raw)
    emotions_path = OUTPUT_DIR / 'emotions.json'
    with open(emotions_path, 'w') as f:
        _json.dump(emotions, f, indent=2)
    print(f'Saved emotions.json ({len(emotions)} timesteps, 6 emotions)')

elif not GROQ_API_KEY:
    print('Skipping emotion analysis — set GROQ_API_KEY to enable')
else:
    print('Skipping emotion analysis (SKIP_EMOTIONS = True)')


## 12. Zip + download

In [ ]:
with timed_phase(bench, "10. Zip output"):
    zip_path = shutil.make_archive("viewer_data", "zip", ".", "viewer_data")

print(f"Created {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

## 13. Benchmark results

In [ ]:
# Print the full summary table
bench.summary()

# Include optimization state in results
results = bench.to_dict()
results["optimizations_enabled"] = ENABLE_OPTIMIZATIONS

# Save to JSON
suffix = "_optimized" if ENABLE_OPTIMIZATIONS else "_baseline"
results_path = Path(f"benchmark_results{suffix}.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {results_path}")
print(f"Optimizations: {'ENABLED' if ENABLE_OPTIMIZATIONS else 'DISABLED'}")
print("\nFull results:")
print(json.dumps(results, indent=2))

## 14. Download results

Downloads both the benchmark JSON and the viewer data zip.

In [ ]:
from google.colab import files

# Download benchmark results
suffix = "_optimized" if ENABLE_OPTIMIZATIONS else "_baseline"
files.download(f"benchmark_results{suffix}.json")

# Download viewer data (same as export notebook)
files.download("viewer_data.zip")